# Tutorial: HLLSet Ring Store

This tutorial demonstrates the **XOR Ring Algebra** architecture for HLLSet storage:

- **Only BASE HLLSets are stored** (leaves in the derivation DAG)
- **All other HLLSets are XOR expressions** of bases (reconstructed on-the-fly)
- **Ring decomposition** finds the unique basis representation
- **W lattice commits** track temporal evolution with base reuse

## Key Principle

Every HLLSet $H$ can be uniquely expressed as XOR of basis elements:

$$H = B_1 \oplus B_2 \oplus \cdots \oplus B_k$$

This expression is stored in the LUT, and $H$ is reconstructed on demand.

In [1]:
import sys
sys.path.insert(0, '..')

import redis
from core.hllset_ring_store import HLLSetRingStore, RingState, DecomposeResult
from core.hllset import HLLSet

# Connect to Redis
r = redis.Redis(host='127.0.0.1', port=6379, db=0)
r.ping()

True

## 1. Initialize Ring Store

In [2]:
# Create store
store = HLLSetRingStore(r, prefix="tutorial:ring:", cache_size=50)

# Create RediSearch index
store.create_index(drop_existing=True)
print("Store initialized")

Store initialized


## 2. Initialize a Ring

A **ring** maintains a set of linearly independent HLLSets (bases).
New HLLSets are decomposed against this basis.

In [3]:
# Initialize ring for our session
ring = store.init_ring("demo:session1", p_bits=10)

print(f"Ring ID: {ring.ring_id}")
print(f"Precision: {ring.p_bits} bits ({1 << ring.p_bits} registers)")
print(f"Initial rank: {ring.rank}")

Ring ID: demo:session1
Precision: 10 bits (1024 registers)
Initial rank: 0


## 3. Ingest Tokens (Auto-Decomposition)

When we ingest a token:
1. HLLSet is created from the token
2. Gaussian elimination checks if it's independent of current basis
3. If independent → added as new BASE (stored)
4. If dependent → stored as XOR expression (not stored as bytes)

In [4]:
# Ingest first token - will be a new base
result1 = store.ingest("demo:session1", "hello", source="demo")

print(f"Token: 'hello'")
print(f"  SHA1: {result1.sha1[:16]}...")
print(f"  Is new base: {result1.is_new_base}")
print(f"  Ring rank: {result1.rank_before} → {result1.rank_after}")

Token: 'hello'
  SHA1: eec13c8eef69646a...
  Is new base: True
  Ring rank: 0 → 0


In [5]:
# Ingest second token - will also be a new base (independent)
result2 = store.ingest("demo:session1", "world", source="demo")

print(f"Token: 'world'")
print(f"  SHA1: {result2.sha1[:16]}...")
print(f"  Is new base: {result2.is_new_base}")
print(f"  Ring rank: {result2.rank_before} → {result2.rank_after}")

Token: 'world'
  SHA1: 05439db54c3c290c...
  Is new base: True
  Ring rank: 0 → 0


In [6]:
# Ingest combined token - may be dependent on existing bases!
# "hello world" creates positions that are union of "hello" and "world"
result3 = store.ingest("demo:session1", "hello world", source="demo")

print(f"Token: 'hello world'")
print(f"  SHA1: {result3.sha1[:16]}...")
print(f"  Is new base: {result3.is_new_base}")
print(f"  Expression: {[s[:12]+'...' for s in result3.expression]}")
print(f"  Ring rank: {result3.rank_before} → {result3.rank_after}")

Token: 'hello world'
  SHA1: 2cdc697d63dbaa8a...
  Is new base: True
  Expression: ['2cdc697d63db...']
  Ring rank: 0 → 0


### Observation

Notice that:
- `"hello"` and `"world"` are **independent bases** (rank increases)
- `"hello world"` might be **dependent** if its positions are expressible as XOR of existing bases

The exact behavior depends on hash collisions and position overlaps.

In [7]:
# Check storage stats
stats = store.stats()
print(f"\nStorage Statistics:")
print(f"  Base HLLSets stored: {stats['base_count']}")
print(f"  Compound HLLSets (XOR expressions): {stats['compound_count']}")
print(f"  Total entries: {stats['total_entries']}")


Storage Statistics:
  Base HLLSets stored: 3
  Compound HLLSets (XOR expressions): 0
  Total entries: 3


## 4. Retrieve HLLSets (XOR Reconstruction)

When we retrieve a compound HLLSet:
1. Look up its XOR expression in the LUT
2. Load all base HLLSets
3. XOR them together
4. Cache the result

In [8]:
# Retrieve all three HLLSets
hll1 = store.get(result1.sha1)
hll2 = store.get(result2.sha1)
hll3 = store.get(result3.sha1)

print(f"Retrieved HLLSets:")
print(f"  'hello':       cardinality ≈ {hll1.cardinality():.1f}")
print(f"  'world':       cardinality ≈ {hll2.cardinality():.1f}")
print(f"  'hello world': cardinality ≈ {hll3.cardinality():.1f}")

Retrieved HLLSets:
  'hello':       cardinality ≈ 2.0
  'world':       cardinality ≈ 2.0
  'hello world': cardinality ≈ 2.0


In [9]:
# Check derivation for each
for sha1, name in [(result1.sha1, 'hello'), (result2.sha1, 'world'), (result3.sha1, 'hello world')]:
    deriv = store.get_derivation(sha1)
    print(f"\n'{name}':")
    print(f"  Operation: {deriv.operation.value}")
    if deriv.operation.value == 'xor':
        print(f"  Bases: {[s[:12]+'...' for s in deriv.bases]}")


'hello':
  Operation: base

'world':
  Operation: base

'hello world':
  Operation: base


## 5. Batch Ingestion

In [10]:
# Ingest a batch of tokens
tokens = ["apple", "banana", "cherry", "date", "elderberry"]
results = store.ingest_batch("demo:session1", tokens, source="fruits")

print("Batch ingestion results:")
for token, result in zip(tokens, results):
    status = "NEW BASE" if result.is_new_base else f"XOR({len(result.expression)} bases)"
    print(f"  '{token}': {status}")

# Check ring rank
ring_state = store.get_ring("demo:session1")
print(f"\nRing rank after batch: {ring_state.rank}")

Batch ingestion results:
  'apple': NEW BASE
  'banana': NEW BASE
  'cherry': NEW BASE
  'date': NEW BASE
  'elderberry': NEW BASE

Ring rank after batch: 0


## 6. W Lattice Commits (Temporal Evolution)

W commits snapshot the ring state at specific time points.
This enables tracking how the basis evolves over time.

In [11]:
# Create first W commit
w0 = store.commit_W("demo:session1", metadata={"phase": "initial"})
print(f"W(0) committed:")
print(f"  Time index: {w0.time_index}")
print(f"  Bases: {len(w0.basis_sha1s)}")

W(0) committed:
  Time index: 0
  Bases: 0


In [12]:
# Ingest more tokens
more_tokens = ["fig", "grape", "honeydew"]
store.ingest_batch("demo:session1", more_tokens, source="fruits2")

# Create second W commit
w1 = store.commit_W("demo:session1", metadata={"phase": "extended"})
print(f"W(1) committed:")
print(f"  Time index: {w1.time_index}")
print(f"  Bases: {len(w1.basis_sha1s)}")

W(1) committed:
  Time index: 1
  Bases: 0


In [13]:
# Compute diff between commits
diff = store.diff_W("demo:session1", t1=0, t2=1)

print(f"\nW Diff (t=0 → t=1):")
print(f"  Added bases: {len(diff.added_bases)}")
print(f"  Removed bases: {len(diff.removed_bases)}")
print(f"  Shared bases: {len(diff.shared_bases)}")
print(f"  Delta rank: {diff.delta_rank:+d}")


W Diff (t=0 → t=1):
  Added bases: 0
  Removed bases: 0
  Shared bases: 0
  Delta rank: +0


## 7. Query by Metadata

In [14]:
# Query only base HLLSets
bases = store.query_bases(limit=10)
print(f"Base HLLSets ({len(bases)} found):")
for meta in bases[:5]:
    print(f"  {meta.sha1[:12]}... source={meta.source} card={meta.cardinality:.1f}")

Base HLLSets (8 found):
  cede032c667c... source=fruits card=2.0
  5da54ef2329f... source=fruits card=2.0
  80e760b93f95... source=fruits card=2.0
  4fcbcfaff1b1... source=fruits card=2.0
  6e677c9403bb... source=fruits card=2.0


In [15]:
# Query by source
fruit_entries = store.query_by_source("fruits")
print(f"\nEntries from source 'fruits': {len(fruit_entries)}")


Entries from source 'fruits': 5


## 8. Eviction (Reference Counting)

Bases with refcount=0 that are not in any active ring can be evicted.

In [16]:
# Check eviction candidates (dry run)
candidates = store.evict_unreferenced(dry_run=True)
print(f"Eviction candidates (dry run): {len(candidates)}")
if candidates:
    for sha1 in candidates[:3]:
        print(f"  {sha1[:16]}...")

Eviction candidates (dry run): 11
  eec13c8eef69646a...
  3e5905979b04fb11...
  49733d5d133aa977...


## 9. Architecture Summary

```
┌─────────────────────────────────────────────────────────────────────────┐
│                      HLLSet Ring Store Architecture                     │
├─────────────────────────────────────────────────────────────────────────┤
│  Persistent (Redis):                                                    │
│    hllring:base:<sha1>     Raw bytes (ONLY for bases)                   │
│    hllring:lut:<sha1>      Derivation: {op: XOR, bases: [...]}          │
│    hllring:meta:<sha1>     Metadata (source, cardinality, etc)          │
│    hllring:ring:<id>       Ring state (basis SHA1s)                     │
│    hllring:W:<id>:<t>      W commit snapshot                            │
├─────────────────────────────────────────────────────────────────────────┤
│  Ephemeral (Session Cache):                                             │
│    LRU cache of reconstructed compounds                                 │
│    In-memory ring matrices (for Gaussian elimination)                   │
└─────────────────────────────────────────────────────────────────────────┘

Key Benefits:
1. Storage efficiency: Only bases stored as bytes
2. Unique representation: XOR decomposition is canonical
3. Temporal tracking: W commits capture evolution
4. Base reuse: Common tokens shared across compounds
```

## 10. Architectural Invariants

### The LUT is the Source of Truth

The critical insight is that **derivations are frozen at ingestion time**:

| Component | Role | Mutability |
|-----------|------|------------|
| **Base HLLSets** | Content-addressed atoms | **Immutable** |
| **LUT entries** | `{op: XOR, bases: [sha1_a, sha1_b, ...]}` | **Immutable** |
| **Ring state** | Current basis for decomposition | Ephemeral |
| **W commits** | Snapshots of ring evolution | Append-only |

When retrieving an HLLSet:
1. Look up derivation in LUT: `{bases: [sha1_a, sha1_b]}`
2. Load those **exact** bases by SHA1
3. XOR them together

The LUT doesn't say "use current ring bases" — it says "use *these exact* bases."

### Temporal Locality ≠ Global Optimality

The W lattice trades **global optimality for temporal locality**:

- **Local decomposition**: Each ring maintains its basis at time $t$. A token at $t_1$ might become a base, while the *same* token at $t_2$ might decompose as XOR of existing bases.
- **"Impure" bases are fine**: A base in $W(t)$ isn't necessarily independent of bases in $W(t')$. But this doesn't matter because all bases exist and are retrievable.
- **Benefits**:
  - $O(\text{rank}^2)$ Gaussian elimination per ingest (not global)
  - No retroactive recomputation when new tokens arrive
  - Parallelizable: different rings process independently
  - W commits are snapshots, not migrations

### Cross-Temporal Queries Work

HLLSets from different W layers can coexist because **all relations are frozen in the LUT**:

```
W(0): bases = {A, B}
W(1): bases = {A, B, C}       # C added
W(2): bases = {A, B, C, D}    # D added

HLLSet X (created at t=1): LUT → X = A ⊕ C
HLLSet Y (created at t=2): LUT → Y = B ⊕ D

# Both retrievable at any time t ≥ 2
```

The ring is a **processing tool** (for decomposition), but the **LUT + bases** are the persistent algebra.

### The Core Invariant

> **Every HLLSet can be reconstructed from its recorded XOR expression, because all referenced bases exist and are immutable.**

This is weaker than "globally minimal basis" but sufficient for correctness and much faster.

In [17]:
# Final stats
final_stats = store.stats()
ring_state = store.get_ring("demo:session1")

print("=" * 50)
print("Final Summary")
print("=" * 50)
print(f"Ring rank (independent bases): {ring_state.rank}")
print(f"Total entries: {final_stats['total_entries']}")
print(f"  - Bases (stored as bytes): {final_stats['base_count']}")
print(f"  - Compounds (XOR expressions): {final_stats['compound_count']}")
print(f"Cache size: {final_stats['cache_size']}")
print(f"W commits: {len(store.list_W_commits('demo:session1'))}")

Final Summary
Ring rank (independent bases): 0
Total entries: 11
  - Bases (stored as bytes): 11
  - Compounds (XOR expressions): 0
Cache size: 3
W commits: 2


## Next Steps

The current implementation uses **Python for ring operations** (Gaussian elimination).
This requires round-trips between Python and Redis.

The next phase will implement **HLLSET.RING.* commands in Rust** for:
- Server-side decomposition (no round-trips)
- Atomic ingest-decompose-store operations
- Efficient basis maintenance

See `docs/HLLSET_RING_MODULE_DESIGN.md` for the Rust module specification.

**Next Steps**

- Add RING commands to existing hllset_rust module
- Implement BitMatrix and Gaussian elimination in Rust
- Update Python HLLSetRingStore to call Rust commands (drop-in replacement)